### Imports

In [1]:
import cv2
import numpy as np
import glob
import time

# 1. Record Photos

In [ ]:
cap = cv2.VideoCapture(0)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

# prepare object points, like (0,0,0), (1,0,0), (2,0,0) ....,(6,5,0)
objp = np.zeros((10*7,3), np.float32)
objp[:,:2] = np.mgrid[0:10,0:7].T.reshape(-1,2)
# compensate for square size
square_size = 0.022
objp *= square_size
 
# Arrays to store object points and image points from all the images.
objpoints = [] # 3d point in real world space
imgpoints = [] # 2d points in image plane.

while True:
    ret, frame = cap.read()
    if (not ret): continue

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    ret, corners = cv2.findChessboardCorners(gray, (10,7), None)

    if ret:
        objpoints.append(objp)

        corners2 = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
        imgpoints.append(corners2)

        cv2.imwrite("images/calib_%f.jpg" % time.time(), frame)
        cv2.drawChessboardCorners(frame, (10,7), corners2, ret)

    cv2.imshow("Calibration", frame)

    key = cv2.waitKey(1)

    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

2026-04-15 18:12:03.255 Python[71632:3398355] WARNING: AVCaptureDeviceTypeExternal is deprecated for Continuity Cameras. Please use AVCaptureDeviceTypeContinuityCamera and add NSCameraUseContinuityCameraDeviceType to your Info.plist.


-1

# 2. Compute points from photos

In [3]:
images = glob.glob('images/*.jpg')

for fname in images:
    img = cv2.imread(fname)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
 
    # Find the chess board corners
    ret, corners = cv2.findChessboardCorners(gray, (10,7), None)
 
    # If found, add object points, image points (after refining them)
    if ret:
        objpoints.append(objp)
 
        corners2 = cv2.cornerSubPix(gray,corners, (11,11), (-1,-1), criteria)
        imgpoints.append(corners2)

# 3. Run Calibration

In [4]:
ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, gray.shape[::-1], None, None)

## 3.1 Optimize Matrix

In [5]:
img = cv2.imread(images[0])
h, w = img.shape[:2]
newcammtx, roi = cv2.getOptimalNewCameraMatrix(mtx, dist, (w,h), 1, (w,h))

## 3.2 Test undistortion

In [ ]:
undistort = False
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if (not ret): continue

    if (undistort):
        frame = cv2.undistort(frame, mtx, dist, None, newcammtx)
        # x, y, w, h = roi
        # frame = frame[]
    
    cv2.imshow("Camera", frame)

    key = cv2.waitKey(1)

    if key == ord('u'):
        undistort = not undistort
    elif key == ord('q'):
        break

cap.release()

cv2.destroyAllWindows()
cv2.waitKey(1)

-1

# 4. Load AR

In [6]:
aruco_dict = cv2.aruco.getPredefinedDictionary(
    cv2.aruco.DICT_4X4_50
)

detector = cv2.aruco.ArucoDetector(aruco_dict)

# 5. Run Detection

In [ ]:
def drawSquare(frame, corners):
    for i in range(4):
        cv2.line(frame, (int(corners[0][i][0]),int(corners[0][i][1])), (int(corners[0][(i+1)%4][0]), int(corners[0][(i+1)%4][1])), (0,0,255), 3)

tagSize = 27 / 1000 # m

cap = cv2.VideoCapture(0)
while True:
    ret, frame = cap.read()
    if (not ret): continue

    corners, ids, _ = detector.detectMarkers(frame)

    objpoints = np.array([
        (-tagSize/2, tagSize/2, 0),
        (tagSize/2, tagSize/2, 0),
        (tagSize/2, -tagSize/2, 0),
        (-tagSize/2, -tagSize/2, 0)
    ])

    # rvecs = []
    # tvecs = []

    for square in corners:
        drawSquare(frame, square)
        ret, rvec, tvec = cv2.solvePnP(objpoints, square, mtx, dist)
        # rvecs.append(rvec)
        # tvecs.append(tvec)
        distance = np.sqrt(tvec[0] ** 2 + tvec[1] ** 2 + tvec[2] ** 2)
        cv2.putText(frame, '%f mm' % (distance * 1000), (int(square[0][0][0]),int(square[0][0][1])), cv2.FONT_HERSHEY_COMPLEX, 1, (255,255,255), 2)

    cv2.imshow('ArUco', frame)

    key = cv2.waitKey(1)

    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)


/var/folders/sn/0sbdmqxx4t94pvtdswvx7jcr0000gn/T/ipykernel_71632/4171985082.py:43: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  cv2.putText(frame, '%f mm' % (distance * 1000), (int(square[0][0][0]),int(square[0][0][1])), cv2.FONT_HERSHEY_COMPLEX, 1, (255,255,255), 2)


-1